In [ ]:
# Install/upgrade the SDK quietly (does nothing if already present)
# %pip -q install --upgrade openai

import os, pathlib, pandas as pd, json, textwrap
from openai import OpenAI
import matplotlib.pyplot as plt


# -------------------------------------------------------------------
# Locate API key: env var ➜ key/openai_key.txt
# -------------------------------------------------------------------
key_path = pathlib.Path("key/openai_key.txt")

if os.getenv("OPENAI_API_KEY") is None and key_path.exists():
    os.environ["OPENAI_API_KEY"] = key_path.read_text().strip()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "No API key found.\n"
        "Create key/openai_key.txt (single line) or export OPENAI_API_KEY in your shell."
    )

client = OpenAI()  # SDK reads the key from the env var

In [ ]:
# data contains the all papers from the JoF and the top 5 journals except jpe that contain the word stock returns or synonyms in the title
# the sampling period is 20 years (7300 days)
# load csv
df = pd.read_csv(r"C:\Users\jonat\Lasso_paper\openalex_abstracts_out\abstracts.csv")
# delete all rows without an abstract - mostly not papers
df = df.dropna(subset=["abstract"])
# test on subsample
df = df.sample(n=10)

In [ ]:
response_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "edges",
        "strict": True,
        "schema": {
            "type": "object", # default (choose from object or array)
            "properties": {
                "edges": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "claim":    {"type": "string"},
                            "source":   {"type": "string"},
                            "sink":     {"type": "string"}
                        },
                        "required": ["claim", "source", "sink"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["edges"],
            "additionalProperties": False
        }
    }
}

system_prompt = (
    "You are an expert annotator of financial research papers."
    "Your task is to read the abstract of a paper and identify any variables that are mentioned as predictors of stock returns."
    "Extract these claims as source (the predictor), sink (stock returns), and the claim itself (the relationship between the two)."
    "Only extract something when the abstract explicitly mentions a predictive relationship with stock returns as a result of the paper."
    "Do not extract a predictive relationship twice."
    "Follow the provided response format strictly and return ONLY valid JSON that matches the provided schema."
)


In [ ]:
def run_extract(row):
    user_prompt = f"Title: {row.title}\nAbstract: {row.abstract}\nExtract edges."

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        response_format=response_format
    )

    return json.loads(response.choices[0].message.content)["edges"]


edges = []
for _, r in df.iterrows():
    for edge in run_extract(r):
        edge["paper_id"] = r.idx
        edges.append(edge)

df_edges = pd.json_normalize(edges)
df_edges

In [73]:
df_edges


,claim,source,sink,paper_id,abstract_x,idx_x,source_idx,abstract_y,idx_y
0,Moral hazard affects asset markets leading to cross-sectional asset pricing anomalies.,moral hazard,stock returns,1,NaN,0,0,"We present a tractable general equilibrium model with multiple sectors in which firms offer workers incentive contracts and simultaneously raise capital in stock markets. Workers optimally invest in the stock market and at the same time hedge labour income risk. Firms rationally take agents' portfolio decisions into account. In equilibrium, the cost of capital of each sector is endogenous. The distortion induced by moral hazard generates counterintuitive effects on the real economy. For example, the value of labour market participation may be higher under moral hazard than under first best, further a positive productivity shock may decrease welfare in the moral hazard economy. In addition, our model generates predictions on the effects of moral hazard on asset markets. For example, in the presence of moral hazard, the capital asset pricing model fails because firms, by choosing optimal incentive contracts, transfer risk both through wages and through the stock market. This leads to several cross-sectional asset pricing “anomalies”, such as size and value effects. As we characterize optimal contracts, we can also present empirical predictions relating workers' compensation, firm productivity, firm size, and financial market abnormal returns.",1
1,Workers' compensation relates to financial market abnormal returns.,workers' compensation,stock returns,1,NaN,1,1,"We present a tractable general equilibrium model with multiple sectors in which firms offer workers incentive contracts and simultaneously raise capital in stock markets. Workers optimally invest in the stock market and at the same time hedge labour income risk. Firms rationally take agents' portfolio decisions into account. In equilibrium, the cost of capital of each sector is endogenous. The distortion induced by moral hazard generates counterintuitive effects on the real economy. For example, the value of labour market participation may be higher under moral hazard than under first best, further a positive productivity shock may decrease welfare in the moral hazard economy. In addition, our model generates predictions on the effects of moral hazard on asset markets. For example, in the presence of moral hazard, the capital asset pricing model fails because firms, by choosing optimal incentive contracts, transfer risk both through wages and through the stock market. This leads to several cross-sectional asset pricing “anomalies”, such as size and value effects. As we characterize optimal contracts, we can also present empirical predictions relating workers' compensation, firm productivity, firm size, and financial market abnormal returns.",1
2,Firm productivity relates to financial market abnormal returns.,firm productivity,stock returns,1,NaN,2,2,"We present a tractable general equilibrium model with multiple sectors in which firms offer workers incentive contracts and simultaneously raise capital in stock markets. Workers optimally invest in the stock market and at the same time hedge labour income risk. Firms rationally take agents' portfolio decisions into account. In equilibrium, the cost of capital of each sector is endogenous. The distortion induced by moral hazard generates counterintuitive effects on the real economy. For example, the value of labour market participation may be higher under moral hazard than under first best, further a positive productivity shock may decrease welfare in the moral hazard economy. In addition, our model generates predictions on the effects of moral hazard on asset markets. For example, in the presence of moral hazard, the capital asset pricing model fails because firms, by choosing optimal incentive contracts, transfer risk both through wages and through the stock market. This leads to several cross-sectional asset pricing “anomalies”

In [72]:
# add source index variable
df_edges["source_idx"] = df_edges.index
# merge abstracts back with results
df_edges = df_edges.merge(
    df[["abstract","idx"]],
    left_on="paper_id",
    right_on="idx",
    how="left"
)

In [ ]:
response_format_topic = {
    "type": "json_schema",
    "json_schema": {
        "name": "topics",
        "strict": True,
        "schema": {
            "type": "object", # default (choose from object or array)
            "properties": {
                "topics": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "topic":    {"type": "string"}
                        },
                        "required": ["topic"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["topics"],
            "additionalProperties": False
        }
    }
}

system_prompt_topic = (
    "You are an expert annotator of financial research papers."
    "Your task is to assing a source variable to exactly ONE topic from a list of 180 topics."
    "Choose the topic that best fits the source variable."
    "If you are unsure, read the provided abstract or context."
    "When no topic fits, assign 'NA'."
    "Follow the provided response format strictly and return ONLY valid JSON that matches the provided schema."
    "Here is the list of topics: [Unions, C-suite, Control stakes, Mutual funds, Venture capital, European sovereign debt, Mining, Company spokesperson, Private / public sector, Pharma, Schools, Russia, Programs / initiatives," 
    "Health insurance, Drexel, Trade agreements, Treasury bonds, Challenges, People familiar, Sales call, Publishing, Financial crisis, Aerospace / defense, Recession, Latin America, Cultural life, SEC, Earnings losses, "
    "Phone companies, Computers, Marketing, Japan, Nuclear / North Korea, NY politics, Tobacco, Product prices, Biology / chemistry / physics, Movie industry, Automotive, Machinery, Bankruptcy, Arts, International exchanges, "
    "Accounting, Space program, Immigration, Small changes, Small possibility, Agreement reached, Oil drilling, Rail / trucking / shipping, Indictments, Positive sentiment, Canada / South Africa, Airlines, California, "
    "Corporate governance, China, Investment banking, Spring / summer, Software, Pensions, Humor / language, Systems, Clintons, Major concerns, Mid-level executives, U.S. Senate, Agriculture, Bank loans, Takeovers, State politics, "
    "Real estate, Futures / indices, Southeast Asia, Optimism, Corrections / amplifications, Government budgets, Exchanges / composites, Currencies / metals, Mortgages, Financial reports, Germany, Rental properties, Committees," 
    "Subsidiaries, Management changes, Share payouts, France / Italy, Acquired investment banks, Credit cards, Bear / bull market, Earnings forecasts, Terrorism, Watchdogs, Oil market, Couriers, Commodities, Utilities, "
    "Foods / consumer goods, Convertible / preferred, Macroeconomic data, Courts, Safety administrations, Reagan, Bush / Obama / Trump, Fees, Gender issues, Trading activity, Microchips, Insurance, Earnings, Luxury / beverages, "
    "Iraq, National security, Buffett, Taxes, Options / VIX, Casinos, Elections, Private equity / hedge funds, Negotiations, European politics, Size, NASD, Mexico, Retail, Long / short term, Wide range, Lawsuits, UK, Revenue growth]"
)

In [ ]:
def run_extract(row):
    user_prompt = f"Source variable: {row.source}\n Abstract: {row.abstract}\n Assign ONE topic."

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt_topic},
            {"role": "user",   "content": user_prompt}
        ],
        response_format=response_format_topic
    )

    return json.loads(response.choices[0].message.content)["topics"]


# Collect edges for all three papers
edges = []
for _, r in df_edges.iterrows():
    for topic in run_extract(r):
        topic["source_idx"] = r.source_idx
        edges.append(topic)

df_edges_topics = pd.json_normalize(edges)
df_edges_topics

In [ ]:
# merge annoation results
final = df_edges_topics.merge(
    df_edges[["source_idx", "source", "sink", "claim", "paper_id"]],
    left_on="source_idx",
    right_on="source_idx",
    how="left"
)


In [ ]:
# Count unique papers per topic
counts = (
    final
    .drop_duplicates(["topic", "paper_id"])
    .groupby("topic")["paper_id"]
    .nunique()
    .sort_values(ascending=False)
)

# remove NA topic
counts = counts[counts.index != "NA"]

# Keep only topics with more than 1 paper
counts = counts[counts > 1]

# Bar plot
plt.figure(figsize=(10, 6))
counts.plot(kind="bar")

plt.xlabel("Topic")
plt.ylabel("Number of Papers")
plt.title("Number of Papers per Topic (≥ 2 Papers)")

plt.tight_layout()
plt.show()
